In [ ]:
import torch 
import torch.nn as nn
import torch.optim as optim
import pandas as pd 


from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader


In [ ]:
df = pd.read_excel(r"C:\\Users\\anton\Desktop\\Raisin_Dataset.xlsx")

X = df.drop(columns= 'Class')
#y - правильные ответы, Kecimen = 1, Besni = 0
y = (df['Class'] == 'Kecimen').astype(int)

In [ ]:
# Откладываем 20% данных для итогового тестирования
# stratify сохраняет доли классов, random_state фиксирует разбиение
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state= 42, stratify= y
)

# Часть оставшихся данных выделяем под валидацию
# Итог: 540 примеров для обучения, 180 для валидации, 180 для теста
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size= 0.25, random_state= 42, stratify= y_train
)


In [ ]:
scaler = StandardScaler()

In [ ]:
# Вычисляем статистики только по обучающим данным и преобразуем их
X_train_scaled = scaler.fit_transform(X_train)

# Применяем сохранённые статистики к тесту и валидации
X_test_scaled = scaler.transform(X_test)
X_val_scaled = scaler.transform(X_val)

In [ ]:
# Переводим признаки в тензоры с числами float32
X_train_t = torch.tensor(X_train_scaled, dtype = torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype = torch.float32)
X_val_t = torch.tensor(X_val_scaled, dtype = torch.float32)

In [ ]:
y_train_t = torch.tensor(y_train.values, dtype = torch.float32).unsqueeze(1) # unsqueeze(1) превращает список меток (N,) в столбец (N, 1)
y_test_t = torch.tensor(y_test.values, dtype = torch.float32).unsqueeze(1)
y_val_t = torch.tensor(y_val.values, dtype = torch.float32).unsqueeze(1)

In [ ]:
class MLP(nn.Module): #  модель с 1 скрытым слоем
    def __init__(self):
        super().__init__()

        # Из 7 признаков вычисляем 32 промежуточных числа
        self.hidden = nn.Linear(7, 32)

        # Заменяем отрицательные значения нулями (функция активации) добавляет нелинейность
        self.relu = nn.ReLU()   # Сначала передавал в ReLU исходный x вместо результата скрытого слоя h.
                                # Исправил на self.relu(h) чтобы данные последовательно проходили слои.
                                # Также разобрался что ReLU сохраняет порядок элементов,
                                # заменяя отрицательные значения нулями

        # Из 32 чисел получаем один логит на изюминку
        self.output = nn.Linear(32, 1)

    
    def forward(self, x):
        h = self.hidden(x)     
        h = self.relu(h)       
        return self.output(h) 



In [ ]:
model = MLP()

In [ ]:
# Adam обновляет веса и смещения, lr задаёт масштаб обновлений
optimizer = optim.Adam(model.parameters(), lr = 0.001)
criterion = nn.BCEWithLogitsLoss()

train_dataset = TensorDataset(X_train_t, y_train_t)

In [ ]:
# Выдаём по 32 примера за бэтч и перемешиваем пары перед каждой эпохой
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle=True)

n_epochs = 100

In [ ]:
# Сила штрафов за величину весов
k_L1 = 0.0001
k_L2 = 0.0001


In [ ]:
# Внешний цикл - эпохи
for epoch in range(n_epochs):
    # Включаем режим обучения
    model.train()

    # Внутренний цикл — батчи текущей эпохи
    for X_batch, y_batch in train_loader:
        # Очищаем градиенты предыдущего батча
        optimizer.zero_grad()

        # Получаем логиты для текущего батча
        logits = model(X_batch)

        # Сравниваем логиты с правильными метками
        loss = criterion(logits, y_batch)

        # L1 - сумма модулей каждого слоя
        abs_weight_hidden_l1 = model.hidden.weight.abs().sum()
        abs_weight_output_l1 = model.output.weight.abs().sum()
                                                                            # Сначала использовал sum(abs(weight)).
                                                                            # Для матрицы это складывало строки и оставляло несколько значений.
                                                                            # Заменил на weight.abs().sum() чтобы получить одно число.
                                                                            # Также разделил коэффициент L1 и рассчитанный штраф,
        # L2 - сумма квадратов каждого слоя
        square_weight_hidden_l2 = model.hidden.weight.square().sum()
        square_weight_output_l2 = model.output.weight.square().sum()

        # Складываем штрафы двух слоёв
        L1_penal = abs_weight_hidden_l1 + abs_weight_output_l1
        L2_penal = square_weight_hidden_l2 + square_weight_output_l2

        # Добавляем штрафы к ошибке классификации
        loss += (k_L1 * L1_penal) + (k_L2 * L2_penal)

        loss.backward()
        optimizer.step()
    model.eval()

    ith torch.no_grad():
        val_logits = model(X_val_t)

        val_loss = criterion(val_logits, y_val_t)

    print(f"Epoch {epoch + 1}, val_loss: {val_loss.item():.4f}")


# Проверяем на тесте модель после последней эпохи
model.eval()

In [ ]:
with torch.no_grad():
    # Получаем по одному логиту на тестовую изюминку
    test_logits = model(X_test_t)

    # Сигмоида переводит логиты в оценки вероятности класса Kecimen
    test_probs = torch.sigmoid(test_logits)

# Вероятность больше 0.5 = 1, иначе 0
preds = (test_probs > 0.5).float()

In [ ]:
# Превращаем тензоры-столбцы в одномерные массивы NumPy
preds_np = preds.numpy().flatten()
y_true = y_test_t.numpy().flatten()
probs_np = test_probs.numpy().flatten()

# Accuracy - доля правильных меток
accuracy = accuracy_score(y_true,preds_np)

roc_auc = roc_auc_score(y_true, probs_np)

print(f"Accuracy: {accuracy:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")